In [3]:
import os
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

import tensorflow as tf
import numpy as np

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.21.0


In [4]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "datasets/archive/PlantVillage",
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "datasets/archive/PlantVillage",
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

class_names = train_ds.class_names

print("Classes:", len(class_names))

Found 20638 files belonging to 15 classes.
Using 16511 files for training.
Found 20638 files belonging to 15 classes.
Using 4127 files for validation.
Classes: 15


In [6]:
image_size=(224,224)

In [7]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

print("Dataset Optimized")

Dataset Optimized


In [8]:
from tensorflow.keras.applications import MobileNetV2

base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

print("MobileNetV2 Loaded Successfully")

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 149s 16us/step
MobileNetV2 Loaded Successfully


In [9]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),

    Dense(256, activation='relu'),
    Dropout(0.3),

    Dense(15, activation='softmax')
])

print("Transfer Learning Model Created")


Transfer Learning Model Created


In [11]:
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 15)             │         3,855 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,589,775 (9.88 MB)

 Trainable params: 331,791 (1.27 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [12]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("MobileNetV2 Compiled Successfully")

MobileNetV2 Compiled Successfully


In [13]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True
)

print("Early Stopping Created")

Early Stopping Created


In [14]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
    "models/best_mobilenetv2.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

print("Checkpoint Created")

Checkpoint Created


In [15]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[
        early_stopping,
        checkpoint
    ]
)

Epoch 1/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 0s 297ms/step - accuracy: 0.4876 - loss: 1.6370
Epoch 1: val_accuracy improved from None to 0.73904, saving model to models/best_mobilenetv2.keras

Epoch 1: finished saving model to models/best_mobilenetv2.keras
516/516 ━━━━━━━━━━━━━━━━━━━━ 195s 372ms/step - accuracy: 0.5926 - loss: 1.2794 - val_accuracy: 0.7390 - val_loss: 0.8036
Epoch 2/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 0s 299ms/step - accuracy: 0.7075 - loss: 0.8759
Epoch 2: val_accuracy improved from 0.73904 to 0.78580, saving model to models/best_mobilenetv2.keras

Epoch 2: finished saving model to models/best_mobilenetv2.keras
516/516 ━━━━━━━━━━━━━━━━━━━━ 194s 377ms/step - accuracy: 0.7166 - loss: 0.8440 - val_accuracy: 0.7858 - val_loss: 0.6519
Epoch 3/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 0s 315ms/step - accuracy: 0.7492 - loss: 0.7438
Epoch 3: val_accuracy improved from 0.78580 to 0.80082, saving model to models/best_mobilenetv2.keras

Epoch 3: finished saving model to models/best_mobilenetv2.ke

In [16]:
base_model.trainable = False

In [17]:
base_model.trainable = True

print("MobileNetV2 Unfrozen")

MobileNetV2 Unfrozen


In [18]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=0.00001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Model Recompiled for Fine-Tuning")


Model Recompiled for Fine-Tuning


In [19]:
fine_tune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=[
        early_stopping,
        checkpoint
    ]
)

Epoch 1/5
516/516 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.3453 - loss: 8.2636
Epoch 1: val_accuracy did not improve from 0.83959
516/516 ━━━━━━━━━━━━━━━━━━━━ 871s 2s/step - accuracy: 0.4998 - loss: 3.8397 - val_accuracy: 0.2435 - val_loss: 9.6703
Epoch 2/5
516/516 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7292 - loss: 0.8727
Epoch 2: val_accuracy did not improve from 0.83959
516/516 ━━━━━━━━━━━━━━━━━━━━ 877s 2s/step - accuracy: 0.7538 - loss: 0.7770 - val_accuracy: 0.6373 - val_loss: 1.8327
Epoch 3/5
516/516 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8242 - loss: 0.5320
Epoch 3: val_accuracy did not improve from 0.83959
516/516 ━━━━━━━━━━━━━━━━━━━━ 878s 2s/step - accuracy: 0.8345 - loss: 0.4984 - val_accuracy: 0.8095 - val_loss: 0.6787
